# IND320 project work, part 2

Viktor, NMBU, autumn 2026.

- GitHub repository: https://github.com/viktorbgulbrandsen/ind320-viktorbgulbrandsen
- Streamlit app: https://ind320-viktorbgulbrandsen.streamlit.app/
- Video: (added when recorded)

# API

I have opted to restructure the repo into projectn/ at root level. Although this might create a bit of hassle in some of the tasks, I think it will yield easier overview of what belongs to which part, and also some practice in modularity.

The first step we take is not using the CSV as previously downloaded but rather directly pulling from the Magasinstatistikk API.


In [1]:
import requests
import pandas as pd

In [2]:
api_url = "https://biapi.nve.no/magasinstatistikk/api/Magasinstatistikk/HentOffentligData"

# just test the hit with a plain GET.

response = requests.get(api_url)
response.raise_for_status()
print(type(response))

<class 'requests.models.Response'>


In [3]:
#reponse OK, traverse with Pandas
df = pd.DataFrame(response.json())
print(df.shape)
df.head()

(14922, 11)


,dato_Id,omrType,omrnr,iso_aar,iso_uke,fyllingsgrad,kapasitet_TWh,fylling_TWh,neste_Publiseringsdato,fyllingsgrad_forrige_uke,endring_fyllingsgrad
0,1995-09-03,EL,4,1995,35,0.944721,21.079208,19.913979,0001-01-01T00:00:00,0.936888,0.007833
1,2020-11-15,EL,1,2020,46,0.974361,6.003264,5.849344,2020-11-25T13:00:00,0.997406,-0.023046
2,2011-08-28,EL,4,2011,34,0.786499,21.079208,16.578772,0001-01-01T00:00:00,0.787193,-0.000694
3,1998-07-05,EL,3,1998,27,0.793969,8.920932,7.082947,0001-01-01T00:00:00,0.733812,0.060157
4,2011-03-27,EL,4,2011,12,0.300820,21.079208,6.341054,0001-01-01T00:00:00,0.311122,-0.010301


# Log

**19 october**

One of the first problems I encountered was that Spark needs an older Python, while I previously deployed and developed with Python 3.14. I asked Claude Fable 5.1 through Claude Code whether Spark really needs 3.12, which answered:

> Not from memory. Let me check what actually ran on 16.10 and what the package itself declares.

It then ran a two-row Spark job on 3.14 and came back with:

> `pyspark` 3.5.9 declares Python `>=3.8` with no upper bound, so nothing warns you. On your Python 3.14, it crashes on the very first 2-row DataFrame: `PicklingError … RecursionError: Stack overflow`. On 16.10 the Cassandra test ran on 3.12, and worked.

When I worried about redeploying the app with a new Python, it answered:

> Maybe not. The app never runs Spark. Spark and Cassandra are notebook-only; the app reads from MongoDB and the NVE API. So streamlit.app can stay on 3.14 as deployed, and only your local env drops to 3.12 for the notebook.

I asked it to set the project to Python 3.12 without forcing it on the deployment, and it answered with:

> `requires-python >=3.12`, `.python-version` 3.12, `requests` + `pymongo` as app deps, `pyspark` + `cassandra-driver` + `setuptools` in dev. Env is now Python 3.12.14 and a 2-row Spark DataFrame runs inside it.
